# 08_01 EES Framework – Prepare Inputs

## Purpose

This notebook prepares the unified input layer for the Explainability-Ethics Scoring (EES) framework.

It consolidates outputs from:

- `03_baseline_modeling`
- `04_shap_interpretation`
- `05_lime_interpretation`
- `06_shap_lime_explaination_disagreement`
- `07_counterfactual_fairness_dice_and_recourse_equity`

Only `ACTIVE_DATASET` should be changed when running this notebook for German Credit, Pima Diabetes or COMPAS.

## EES Input Layer

This notebook does **not** calculate the final EES score.

It creates a clean, model-level and instance-level input layer for later notebooks.

The revised designs are supported:

### Fidelity

```text
1/3 SHAP Reconstruction Fidelity
+ 1/3 LIME Local Surrogate Fidelity
+ 1/3 SHAP-LIME Agreement Score
```

### Actionability

```text
1/3 Availability
+ 1/3 Numeric Proximity
+ 1/3 Sparsity
```

In [1]:
# =========================================================
# 0. Dataset Configuration
# =========================================================

ACTIVE_DATASET = "german_credit"
# ACTIVE_DATASET = "pima_diabetes"
# ACTIVE_DATASET = "compas"

DATASET_CONFIGS = {
    "german_credit": {
        "dataset_name": "german_credit",
        "dataset_label": "German Credit",
        "target_name": "target",
        "positive_class_label": "Default",
        "negative_class_label": "Non-default",
        "results_folder": "german_credit",
        "preprocessing_folder": "02_data_preprocessing_german_credit",
        "baseline_folder": "03_baseline_modeling_german_credit",
        "shap_folder": "04_shap_interpretation_german_credit",
        "lime_folder": "05_lime_interpretation_german_credit",
        "disagreement_folder": "06_shap_lime_explaination_disagreement_german_credit",
        "counterfactual_folder": "07_counterfactual_fairness_dice_and_recourse_equity_german_credit",
        "ees_input_folder": "08_01_ees_framework_prepare_inputs_german_credit",
        "unfavourable_class": 1,
        "desired_class": 0,
        "immutable_features": ["Age", "Sex"]
    },
    "pima_diabetes": {
        "dataset_name": "pima_diabetes",
        "dataset_label": "Pima Diabetes",
        "target_name": "target",
        "positive_class_label": "Diabetes",
        "negative_class_label": "No diabetes",
        "results_folder": "pima_diabetes",
        "preprocessing_folder": "02_data_preprocessing_pima_diabetes",
        "baseline_folder": "03_baseline_modeling_pima_diabetes",
        "shap_folder": "04_shap_interpretation_pima_diabetes",
        "lime_folder": "05_lime_interpretation_pima_diabetes",
        "disagreement_folder": "06_shap_lime_explaination_disagreement_pima_diabetes",
        "counterfactual_folder": "07_counterfactual_fairness_dice_and_recourse_equity_pima_diabetes",
        "ees_input_folder": "08_01_ees_framework_prepare_inputs_pima_diabetes",
        "unfavourable_class": 1,
        "desired_class": 0,
        "immutable_features": ["Age"]
    },
    "compas": {
        "dataset_name": "compas",
        "dataset_label": "COMPAS",
        "target_name": "target",
        "positive_class_label": "Recidivism",
        "negative_class_label": "No recidivism",
        "results_folder": "compas",
        "preprocessing_folder": "02_data_preprocessing_compas",
        "baseline_folder": "03_baseline_modeling_compas",
        "shap_folder": "04_shap_interpretation_compas",
        "lime_folder": "05_lime_interpretation_compas",
        "disagreement_folder": "06_shap_lime_explaination_disagreement_compas",
        "counterfactual_folder": "07_counterfactual_fairness_dice_and_recourse_equity_compas",
        "ees_input_folder": "08_01_ees_framework_prepare_inputs_compas",
        "unfavourable_class": 1,
        "desired_class": 0,
        "immutable_features": ["age", "sex", "race"]
    }
}

CONFIG = DATASET_CONFIGS[ACTIVE_DATASET]

print("Active dataset:", CONFIG["dataset_label"])
print("Dataset name:", CONFIG["dataset_name"])

Active dataset: German Credit
Dataset name: german_credit


In [2]:
# =========================================================
# 1. Import Required Libraries
# =========================================================

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
from datetime import datetime
import json
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 250)
pd.set_option("display.float_format", lambda x: f"{x:.6f}")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Libraries imported successfully.")

Libraries imported successfully.


In [3]:
# =========================================================
# 2. Mount Google Drive and Configure Paths
# =========================================================

from google.colab import drive

try:
    drive.mount('/content/drive')
except Exception as e:
    print("Drive mount skipped or already available:", e)

ROOT_PATH = Path("/content/drive/MyDrive/Master_Research")

DATASET_NAME = CONFIG["dataset_name"]
NOTEBOOK_NAME = CONFIG["ees_input_folder"]

RESULTS_ROOT = ROOT_PATH / "07_Results" / CONFIG["results_folder"]

PREPROCESSING_RESULTS_PATH = RESULTS_ROOT / CONFIG["preprocessing_folder"]
PREPROCESSING_DATASETS_PATH = PREPROCESSING_RESULTS_PATH / "datasets"

BASELINE_RESULTS_PATH = RESULTS_ROOT / CONFIG["baseline_folder"]
BASELINE_TABLES_PATH = BASELINE_RESULTS_PATH / "tables"
BASELINE_DATASETS_PATH = BASELINE_RESULTS_PATH / "datasets"

SHAP_RESULTS_PATH = RESULTS_ROOT / CONFIG["shap_folder"]
SHAP_TABLES_PATH = SHAP_RESULTS_PATH / "tables"
SHAP_DATASETS_PATH = SHAP_RESULTS_PATH / "datasets"

LIME_RESULTS_PATH = RESULTS_ROOT / CONFIG["lime_folder"]
LIME_TABLES_PATH = LIME_RESULTS_PATH / "tables"
LIME_DATASETS_PATH = LIME_RESULTS_PATH / "datasets"

DISAGREEMENT_RESULTS_PATH = RESULTS_ROOT / CONFIG["disagreement_folder"]
DISAGREEMENT_TABLES_PATH = DISAGREEMENT_RESULTS_PATH / "tables"
DISAGREEMENT_DATASETS_PATH = DISAGREEMENT_RESULTS_PATH / "datasets"

COUNTERFACTUAL_RESULTS_PATH = RESULTS_ROOT / CONFIG["counterfactual_folder"]
COUNTERFACTUAL_TABLES_PATH = COUNTERFACTUAL_RESULTS_PATH / "tables"
COUNTERFACTUAL_DATASETS_PATH = COUNTERFACTUAL_RESULTS_PATH / "datasets"

RESULTS_PATH = RESULTS_ROOT / NOTEBOOK_NAME
TABLES_PATH = RESULTS_PATH / "tables"
DATASETS_PATH = RESULTS_PATH / "datasets"
LOGS_PATH = RESULTS_PATH / "logs"

for path in [RESULTS_PATH, TABLES_PATH, DATASETS_PATH, LOGS_PATH]:
    path.mkdir(parents=True, exist_ok=True)

print("Results root:", RESULTS_ROOT)
print("EES input output path:", RESULTS_PATH)

Mounted at /content/drive
Results root: /content/drive/MyDrive/Master_Research/07_Results/german_credit
EES input output path: /content/drive/MyDrive/Master_Research/07_Results/german_credit/08_01_ees_framework_prepare_inputs_german_credit


In [4]:
# =========================================================
# 3. Helper Functions
# =========================================================

def read_csv_safely(path, label, required=True):
    path = Path(path)
    if not path.exists():
        if required:
            raise FileNotFoundError(f"{label} not found at: {path}")
        print(f"Optional file not found: {label} at {path}")
        return pd.DataFrame()
    return pd.read_csv(path)


def find_first_existing_file(candidate_paths, label, required=True):
    for path in candidate_paths:
        path = Path(path)
        if path.exists():
            print(f"{label} found at: {path}")
            return path
    if required:
        checked = "\n".join(str(path) for path in candidate_paths)
        raise FileNotFoundError(f"{label} not found. Checked:\n{checked}")
    print(f"{label} not found.")
    return None


def save_df(df, filename, folder=None, index=False):
    if folder is None:
        folder = TABLES_PATH
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    output_path = folder / filename
    df.to_csv(output_path, index=index)
    print(f"Saved: {output_path}")
    return output_path


def save_excel(dfs, filename, folder=None):
    if folder is None:
        folder = TABLES_PATH
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    output_path = folder / filename
    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        for sheet_name, df in dfs.items():
            if isinstance(df, pd.DataFrame):
                df.to_excel(writer, sheet_name=sheet_name[:31], index=False)
    print(f"Saved: {output_path}")
    return output_path


def add_dataset_metadata(df):
    if df.empty:
        return df
    out = df.copy()
    if "dataset_name" not in out.columns:
        out.insert(0, "dataset_name", DATASET_NAME)
    if "dataset_label" not in out.columns:
        out.insert(1, "dataset_label", CONFIG["dataset_label"])
    return out

## 4. Load Core Upstream Outputs

In [5]:
# =========================================================
# 4. Load Core Upstream Outputs
# =========================================================

baseline_performance_file = find_first_existing_file([BASELINE_DATASETS_PATH / "baseline_model_performance.csv", BASELINE_TABLES_PATH / "baseline_model_performance.csv"], "Baseline model performance")
baseline_predictions_file = find_first_existing_file([BASELINE_DATASETS_PATH / "baseline_prediction_summary.csv", BASELINE_TABLES_PATH / "baseline_prediction_summary.csv"], "Baseline prediction summary", required=False)
model_registry_file = find_first_existing_file([BASELINE_DATASETS_PATH / "model_registry.csv", BASELINE_TABLES_PATH / "model_registry.csv"], "Model registry", required=False)

shap_long_file = find_first_existing_file([SHAP_DATASETS_PATH / "all_test_local_shap_long.csv", SHAP_TABLES_PATH / "all_test_local_shap_long.csv", SHAP_DATASETS_PATH / "shap_all_test_long.csv", SHAP_TABLES_PATH / "shap_all_test_long.csv"], "SHAP long output")
shap_reconstruction_file = find_first_existing_file([SHAP_DATASETS_PATH / "shap_reconstruction_diagnostics.csv", SHAP_TABLES_PATH / "shap_reconstruction_diagnostics.csv", SHAP_DATASETS_PATH / "shap_fidelity_inputs.csv", SHAP_TABLES_PATH / "shap_fidelity_inputs.csv"], "SHAP reconstruction diagnostics")

lime_long_file = find_first_existing_file([LIME_DATASETS_PATH / "lime_all_test_long.csv", LIME_TABLES_PATH / "lime_all_test_long.csv"], "LIME long output")
lime_fidelity_file = find_first_existing_file([LIME_DATASETS_PATH / "lime_fidelity_inputs.csv", LIME_TABLES_PATH / "lime_fidelity_inputs.csv"], "LIME fidelity inputs")

agreement_file = find_first_existing_file([DISAGREEMENT_DATASETS_PATH / "shap_lime_agreement_for_fidelity.csv", DISAGREEMENT_TABLES_PATH / "shap_lime_agreement_for_fidelity.csv", DISAGREEMENT_DATASETS_PATH / "model_level_shap_lime_agreement_summary.csv", DISAGREEMENT_TABLES_PATH / "model_level_shap_lime_agreement_summary.csv"], "SHAP-LIME agreement for fidelity")

actionability_file = find_first_existing_file([COUNTERFACTUAL_DATASETS_PATH / "ees_actionability_model_scores.csv", COUNTERFACTUAL_TABLES_PATH / "ees_actionability_model_scores.csv", COUNTERFACTUAL_DATASETS_PATH / "actionability_score_summary.csv", COUNTERFACTUAL_TABLES_PATH / "actionability_score_summary.csv"], "Actionability model scores", required=False)

baseline_performance_df = read_csv_safely(baseline_performance_file, "baseline_model_performance")
baseline_prediction_summary_df = read_csv_safely(baseline_predictions_file, "baseline_prediction_summary", required=False) if baseline_predictions_file else pd.DataFrame()
model_registry_df = read_csv_safely(model_registry_file, "model_registry", required=False) if model_registry_file else pd.DataFrame()
shap_long_df = read_csv_safely(shap_long_file, "SHAP long output")
shap_reconstruction_diagnostics_df = read_csv_safely(shap_reconstruction_file, "SHAP reconstruction diagnostics")
lime_long_df = read_csv_safely(lime_long_file, "LIME long output")
lime_fidelity_inputs_df = read_csv_safely(lime_fidelity_file, "LIME fidelity inputs")
shap_lime_agreement_for_fidelity_df = read_csv_safely(agreement_file, "SHAP-LIME agreement for fidelity")
actionability_model_scores_df = read_csv_safely(actionability_file, "Actionability model scores", required=False) if actionability_file else pd.DataFrame()

print("baseline_performance_df:", baseline_performance_df.shape)
print("baseline_prediction_summary_df:", baseline_prediction_summary_df.shape)
print("model_registry_df:", model_registry_df.shape)
print("shap_long_df:", shap_long_df.shape)
print("shap_reconstruction_diagnostics_df:", shap_reconstruction_diagnostics_df.shape)
print("lime_long_df:", lime_long_df.shape)
print("lime_fidelity_inputs_df:", lime_fidelity_inputs_df.shape)
print("shap_lime_agreement_for_fidelity_df:", shap_lime_agreement_for_fidelity_df.shape)
print("actionability_model_scores_df:", actionability_model_scores_df.shape)

Baseline model performance found at: /content/drive/MyDrive/Master_Research/07_Results/german_credit/03_baseline_modeling_german_credit/datasets/baseline_model_performance.csv
Baseline prediction summary found at: /content/drive/MyDrive/Master_Research/07_Results/german_credit/03_baseline_modeling_german_credit/datasets/baseline_prediction_summary.csv
Model registry found at: /content/drive/MyDrive/Master_Research/07_Results/german_credit/03_baseline_modeling_german_credit/datasets/model_registry.csv
SHAP long output found at: /content/drive/MyDrive/Master_Research/07_Results/german_credit/04_shap_interpretation_german_credit/datasets/all_test_local_shap_long.csv
SHAP reconstruction diagnostics found at: /content/drive/MyDrive/Master_Research/07_Results/german_credit/04_shap_interpretation_german_credit/datasets/shap_reconstruction_diagnostics.csv
LIME long output found at: /content/drive/MyDrive/Master_Research/07_Results/german_credit/05_lime_interpretation_german_credit/datasets/lim

## 5. Validate Input Structure

In [6]:
# =========================================================
# 5. Validate Input Structure
# =========================================================

if "shap_lime_agreement_score" not in shap_lime_agreement_for_fidelity_df.columns:
    if "mean_agreement_score" in shap_lime_agreement_for_fidelity_df.columns:
        shap_lime_agreement_for_fidelity_df = shap_lime_agreement_for_fidelity_df.rename(columns={"mean_agreement_score": "shap_lime_agreement_score"})

checks = []

def check_cols(name, df, cols):
    checks.append({"check": f"{name} output not empty", "passed": not df.empty})
    for col in cols:
        checks.append({"check": f"{name} has {col}", "passed": col in df.columns})

check_cols("baseline_performance", baseline_performance_df, ["model_id", "model_name"])
check_cols("shap_long", shap_long_df, ["model_id", "model_name", "test_instance_position", "feature", "abs_shap_value"])
check_cols("lime_long", lime_long_df, ["model_id", "model_name", "test_instance_position", "feature", "abs_lime_weight"])
check_cols("shap_reconstruction", shap_reconstruction_diagnostics_df, ["model_id", "model_name", "test_instance_position", "reconstruction_error"])
check_cols("lime_fidelity", lime_fidelity_inputs_df, ["model_id", "model_name", "test_instance_position", "lime_r2_score"])
check_cols("shap_lime_agreement", shap_lime_agreement_for_fidelity_df, ["model_id", "model_name", "shap_lime_agreement_score"])

if not actionability_model_scores_df.empty:
    check_cols("actionability", actionability_model_scores_df, ["model_id", "model_name", "actionability_score"])

ees_input_validation_df = pd.DataFrame(checks)
display(ees_input_validation_df)
save_df(ees_input_validation_df, "ees_input_validation.csv")

if not ees_input_validation_df["passed"].all():
    raise ValueError("One or more EES input validation checks failed.")

,check,passed
0,baseline_performance output not empty,True
1,baseline_performance has model_id,True
2,baseline_performance has model_name,True
3,shap_long output not empty,True
4,shap_long has model_id,True
5,shap_long has model_name,True
6,shap_long has test_instance_position,True
7,shap_long has feature,True
8,shap_long has abs_shap_value,True
9,lime_long output not empty,True


Saved: /content/drive/MyDrive/Master_Research/07_Results/german_credit/08_01_ees_framework_prepare_inputs_german_credit/tables/ees_input_validation.csv


## 6. Create Master Model Index

In [7]:
# =========================================================
# 6. Create Master Model Index
# =========================================================

model_sources = []
for source_name, df in [("baseline", baseline_performance_df), ("shap", shap_long_df), ("lime", lime_long_df), ("agreement", shap_lime_agreement_for_fidelity_df), ("actionability", actionability_model_scores_df)]:
    if not df.empty and {"model_id", "model_name"}.issubset(df.columns):
        temp = df[["model_id", "model_name"]].drop_duplicates().copy()
        temp["source"] = source_name
        model_sources.append(temp)

model_source_registry_df = pd.concat(model_sources, ignore_index=True) if model_sources else pd.DataFrame(columns=["model_id", "model_name", "source"])

master_model_index_df = model_source_registry_df[["model_id", "model_name"]].drop_duplicates().sort_values("model_id").reset_index(drop=True)
master_model_index_df["dataset_name"] = DATASET_NAME
master_model_index_df["dataset_label"] = CONFIG["dataset_label"]
master_model_index_df["positive_class_label"] = CONFIG["positive_class_label"]
master_model_index_df["negative_class_label"] = CONFIG["negative_class_label"]

display(master_model_index_df)
display(model_source_registry_df)
save_df(master_model_index_df, "master_model_index.csv")
save_df(model_source_registry_df, "model_source_registry.csv")

,model_id,model_name,dataset_name,dataset_label,positive_class_label,negative_class_label
0,decision_tree,Decision Tree,german_credit,German Credit,Default,Non-default
1,logistic_regression,Logistic Regression,german_credit,German Credit,Default,Non-default
2,mlp,MLP,german_credit,German Credit,Default,Non-default
3,random_forest,Random Forest,german_credit,German Credit,Default,Non-default
4,xgboost,XGBoost,german_credit,German Credit,Default,Non-default


,model_id,model_name,source
0,xgboost,XGBoost,baseline
1,decision_tree,Decision Tree,baseline
2,mlp,MLP,baseline
3,random_forest,Random Forest,baseline
4,logistic_regression,Logistic Regression,baseline
5,logistic_regression,Logistic Regression,shap
6,decision_tree,Decision Tree,shap
7,random_forest,Random Forest,shap
8,xgboost,XGBoost,shap
9,mlp,MLP,shap


Saved: /content/drive/MyDrive/Master_Research/07_Results/german_credit/08_01_ees_framework_prepare_inputs_german_credit/tables/master_model_index.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/german_credit/08_01_ees_framework_prepare_inputs_german_credit/tables/model_source_registry.csv


PosixPath('/content/drive/MyDrive/Master_Research/07_Results/german_credit/08_01_ees_framework_prepare_inputs_german_credit/tables/model_source_registry.csv')

## 7. Prepare EES Input Tables

In [8]:
# =========================================================
# 7. Prepare EES Input Tables
# =========================================================

ees_model_performance_input_df = add_dataset_metadata(baseline_performance_df.copy())
ees_shap_long_input_df = add_dataset_metadata(shap_long_df.copy())
ees_shap_reconstruction_input_df = add_dataset_metadata(shap_reconstruction_diagnostics_df.copy())
ees_lime_long_input_df = add_dataset_metadata(lime_long_df.copy())
ees_lime_fidelity_input_df = add_dataset_metadata(lime_fidelity_inputs_df.copy())

ees_shap_lime_agreement_input_df = shap_lime_agreement_for_fidelity_df.copy()
if "shap_lime_agreement_score_pct" not in ees_shap_lime_agreement_input_df.columns:
    ees_shap_lime_agreement_input_df["shap_lime_agreement_score_pct"] = ees_shap_lime_agreement_input_df["shap_lime_agreement_score"] * 100
ees_shap_lime_agreement_input_df = add_dataset_metadata(ees_shap_lime_agreement_input_df)

ees_actionability_input_df = add_dataset_metadata(actionability_model_scores_df.copy()) if not actionability_model_scores_df.empty else pd.DataFrame()

for df, name in [
    (ees_model_performance_input_df, "ees_model_performance_input.csv"),
    (ees_shap_long_input_df, "ees_shap_long_input.csv"),
    (ees_shap_reconstruction_input_df, "ees_shap_reconstruction_input.csv"),
    (ees_lime_long_input_df, "ees_lime_long_input.csv"),
    (ees_lime_fidelity_input_df, "ees_lime_fidelity_input.csv"),
    (ees_shap_lime_agreement_input_df, "ees_shap_lime_agreement_input.csv"),
    (ees_actionability_input_df, "ees_actionability_input.csv")
]:
    save_df(df, name)
    save_df(df, name, folder=DATASETS_PATH)

display(ees_model_performance_input_df.head())
display(ees_shap_lime_agreement_input_df)
display(ees_actionability_input_df)

Saved: /content/drive/MyDrive/Master_Research/07_Results/german_credit/08_01_ees_framework_prepare_inputs_german_credit/tables/ees_model_performance_input.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/german_credit/08_01_ees_framework_prepare_inputs_german_credit/datasets/ees_model_performance_input.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/german_credit/08_01_ees_framework_prepare_inputs_german_credit/tables/ees_shap_long_input.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/german_credit/08_01_ees_framework_prepare_inputs_german_credit/datasets/ees_shap_long_input.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/german_credit/08_01_ees_framework_prepare_inputs_german_credit/tables/ees_shap_reconstruction_input.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/german_credit/08_01_ees_framework_prepare_inputs_german_credit/datasets/ees_shap_reconstruction_input.csv
Saved: /content/drive/MyDrive/Master_Research/07_Res

,dataset_name,dataset_label,model_id,model_name,training_time_seconds,accuracy,precision,recall,f1_score,roc_auc,average_precision,true_negative,false_positive,false_negative,true_positive,false_positive_rate,false_negative_rate,business_priority_score,business_priority_score_pct,business_priority_rank
0,german_credit,German Credit,xgboost,XGBoost,0.122750,0.765000,0.632653,0.516667,0.568807,0.781429,0.656290,122,18,29,31,0.128571,0.483333,0.619293,61.929252,1
1,german_credit,German Credit,decision_tree,Decision Tree,0.006912,0.680000,0.470588,0.533333,0.500000,0.739940,0.551867,104,36,28,32,0.257143,0.466667,0.582766,58.276646,2
2,german_credit,German Credit,mlp,MLP,2.513024,0.710000,0.517857,0.483333,0.500000,0.740238,0.552379,113,27,31,29,0.192857,0.516667,0.567310,56.730952,3
3,german_credit,German Credit,random_forest,Random Forest,0.777399,0.745000,0.615385,0.400000,0.484848,0.769167,0.618334,125,15,36,24,0.107143,0.600000,0.553827,55.382692,4
4,german_credit,German Credit,logistic_regression,Logistic Regression,0.018544,0.735000,0.589744,0.383333,0.464646,0.761429,0.617207,124,16,37,23,0.114286,0.616667,0.538044,53.804396,5


,dataset_name,dataset_label,model_id,model_name,n_instances,mean_kendall_tau,shap_lime_agreement_score,shap_lime_agreement_score_pct,shap_lime_agreement_rank,shap_lime_agreement_band,mean_top_k_overlap_ratio
0,german_credit,German Credit,random_forest,Random Forest,200,0.177347,0.588673,58.867329,1,Moderate agreement,0.604500
1,german_credit,German Credit,decision_tree,Decision Tree,200,0.149506,0.574753,57.475288,2,Moderate agreement,0.745500
2,german_credit,German Credit,xgboost,XGBoost,200,0.102219,0.551109,55.110948,3,Moderate agreement,0.668000
3,german_credit,German Credit,logistic_regression,Logistic Regression,200,0.073769,0.536884,53.688425,4,Weak agreement,0.515500
4,german_credit,German Credit,mlp,MLP,200,-0.304491,0.347755,34.775466,5,Poor agreement,0.386500


,dataset_name,dataset_label,model_id,model_name,n_unfavourable_instances,n_skipped_due_to_missing_values,n_actionable_recourse_instances,mean_min_normalized_numeric_distance,median_min_normalized_numeric_distance,mean_changed_features,median_changed_features,availability_score,proximity_score,sparsity_score,actionability_score,actionability_score_pct,actionability_rank,actionability_band
0,german_credit,German Credit,decision_tree,Decision Tree,68,0,68,0.025291,0.007344,1.382353,1.000000,1.000000,0.974709,0.846405,0.940371,94.037147,1,High actionability
1,german_credit,German Credit,xgboost,XGBoost,49,0,49,0.044535,0.035714,1.265306,1.000000,1.000000,0.955465,0.859410,0.938292,93.829177,2,High actionability
2,german_credit,German Credit,random_forest,Random Forest,39,0,39,0.036329,0.023438,1.512821,2.000000,1.000000,0.963671,0.831909,0.931860,93.185992,3,High actionability
3,german_credit,German Credit,mlp,MLP,56,0,56,0.053485,0.044850,1.375000,1.000000,1.000000,0.946515,0.847222,0.931246,93.124584,4,High actionability
4,german_credit,German Credit,logistic_regression,Logistic Regression,39,0,39,0.046948,0.025879,1.641026,2.000000,1.000000,0.953052,0.817664,0.923572,92.357181,5,High actionability


## 8. Create Master EES Input Table

In [9]:
# =========================================================
# 8. Create Master EES Input Table
# =========================================================

master_ees_input_df = master_model_index_df.copy()

baseline_cols = [col for col in ["model_id", "accuracy", "precision", "recall", "f1_score", "roc_auc", "average_precision", "business_priority_score", "business_priority_rank"] if col in baseline_performance_df.columns]
if len(baseline_cols) > 1:
    master_ees_input_df = master_ees_input_df.merge(baseline_performance_df[baseline_cols], on="model_id", how="left")

agreement_cols = [col for col in ["model_id", "shap_lime_agreement_score", "shap_lime_agreement_score_pct", "mean_kendall_tau", "mean_top_k_overlap_ratio"] if col in shap_lime_agreement_for_fidelity_df.columns]
if len(agreement_cols) > 1:
    master_ees_input_df = master_ees_input_df.merge(shap_lime_agreement_for_fidelity_df[agreement_cols], on="model_id", how="left")

if not actionability_model_scores_df.empty:
    actionability_cols = [col for col in ["model_id", "availability_score", "proximity_score", "sparsity_score", "actionability_score", "actionability_rank", "mean_min_normalized_numeric_distance", "mean_changed_features"] if col in actionability_model_scores_df.columns]
    if len(actionability_cols) > 1:
        master_ees_input_df = master_ees_input_df.merge(actionability_model_scores_df[actionability_cols], on="model_id", how="left")

display(master_ees_input_df)
save_df(master_ees_input_df, "master_ees_input.csv")
save_df(master_ees_input_df, "master_ees_input.csv", folder=DATASETS_PATH)

,model_id,model_name,dataset_name,dataset_label,positive_class_label,negative_class_label,accuracy,precision,recall,f1_score,roc_auc,average_precision,business_priority_score,business_priority_rank,shap_lime_agreement_score,shap_lime_agreement_score_pct,mean_kendall_tau,mean_top_k_overlap_ratio,availability_score,proximity_score,sparsity_score,actionability_score,actionability_rank,mean_min_normalized_numeric_distance,mean_changed_features
0,decision_tree,Decision Tree,german_credit,German Credit,Default,Non-default,0.680000,0.470588,0.533333,0.500000,0.739940,0.551867,0.582766,2,0.574753,57.475288,0.149506,0.745500,1.000000,0.974709,0.846405,0.940371,1,0.025291,1.382353
1,logistic_regression,Logistic Regression,german_credit,German Credit,Default,Non-default,0.735000,0.589744,0.383333,0.464646,0.761429,0.617207,0.538044,5,0.536884,53.688425,0.073769,0.515500,1.000000,0.953052,0.817664,0.923572,5,0.046948,1.641026
2,mlp,MLP,german_credit,German Credit,Default,Non-default,0.710000,0.517857,0.483333,0.500000,0.740238,0.552379,0.567310,3,0.347755,34.775466,-0.304491,0.386500,1.000000,0.946515,0.847222,0.931246,4,0.053485,1.375000
3,random_forest,Random Forest,german_credit,German Credit,Default,Non-default,0.745000,0.615385,0.400000,0.484848,0.769167,0.618334,0.553827,4,0.588673,58.867329,0.177347,0.604500,1.000000,0.963671,0.831909,0.931860,3,0.036329,1.512821
4,xgboost,XGBoost,german_credit,German Credit,Default,Non-default,0.765000,0.632653,0.516667,0.568807,0.781429,0.656290,0.619293,1,0.551109,55.110948,0.102219,0.668000,1.000000,0.955465,0.859410,0.938292,2,0.044535,1.265306


Saved: /content/drive/MyDrive/Master_Research/07_Results/german_credit/08_01_ees_framework_prepare_inputs_german_credit/tables/master_ees_input.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/german_credit/08_01_ees_framework_prepare_inputs_german_credit/datasets/master_ees_input.csv


PosixPath('/content/drive/MyDrive/Master_Research/07_Results/german_credit/08_01_ees_framework_prepare_inputs_german_credit/datasets/master_ees_input.csv')

## 9. Output Inventory

In [10]:
# =========================================================
# 9. Output Inventory
# =========================================================

ees_input_output_inventory_df = pd.DataFrame([
    {"output_file": "master_ees_input.csv", "description": "Model-level master input table for EES scoring.", "used_by": "08_02 to 08_07"},
    {"output_file": "ees_model_performance_input.csv", "description": "Baseline model performance input.", "used_by": "EES scoring and consolidation"},
    {"output_file": "ees_shap_reconstruction_input.csv", "description": "SHAP reconstruction diagnostics for Fidelity.", "used_by": "08_02 Fidelity"},
    {"output_file": "ees_lime_fidelity_input.csv", "description": "LIME local surrogate fidelity input.", "used_by": "08_02 Fidelity"},
    {"output_file": "ees_shap_lime_agreement_input.csv", "description": "SHAP-LIME agreement score for revised Fidelity pillar.", "used_by": "08_02 Fidelity"},
    {"output_file": "ees_actionability_input.csv", "description": "Three-pillar actionability model-level input.", "used_by": "08_04 Actionability"}
])

display(ees_input_output_inventory_df)
save_df(ees_input_output_inventory_df, "ees_input_output_inventory.csv")

,output_file,description,used_by
0,master_ees_input.csv,Model-level master input table for EES scoring.,08_02 to 08_07
1,ees_model_performance_input.csv,Baseline model performance input.,EES scoring and consolidation
2,ees_shap_reconstruction_input.csv,SHAP reconstruction diagnostics for Fidelity.,08_02 Fidelity
3,ees_lime_fidelity_input.csv,LIME local surrogate fidelity input.,08_02 Fidelity
4,ees_shap_lime_agreement_input.csv,SHAP-LIME agreement score for revised Fidelity...,08_02 Fidelity
5,ees_actionability_input.csv,Three-pillar actionability model-level input.,08_04 Actionability


Saved: /content/drive/MyDrive/Master_Research/07_Results/german_credit/08_01_ees_framework_prepare_inputs_german_credit/tables/ees_input_output_inventory.csv


PosixPath('/content/drive/MyDrive/Master_Research/07_Results/german_credit/08_01_ees_framework_prepare_inputs_german_credit/tables/ees_input_output_inventory.csv')

## 10. Save Workbook and Manifest

In [11]:
# =========================================================
# 10. Save Workbook and Manifest
# =========================================================

save_excel(
    {
        "master_ees_input": master_ees_input_df,
        "model_performance": ees_model_performance_input_df,
        "shap_reconstruction": ees_shap_reconstruction_input_df,
        "lime_fidelity": ees_lime_fidelity_input_df,
        "shap_lime_agreement": ees_shap_lime_agreement_input_df,
        "actionability": ees_actionability_input_df,
        "model_index": master_model_index_df,
        "validation": ees_input_validation_df,
        "output_inventory": ees_input_output_inventory_df
    },
    f"ees_framework_prepare_inputs_{DATASET_NAME}_outputs.xlsx"
)

manifest = {
    "dataset_name": DATASET_NAME,
    "dataset_label": CONFIG["dataset_label"],
    "notebook_name": NOTEBOOK_NAME,
    "run_timestamp": datetime.now().isoformat(),
    "random_state": RANDOM_STATE,
    "positive_class_label": CONFIG["positive_class_label"],
    "negative_class_label": CONFIG["negative_class_label"],
    "fidelity_design": "Equal-weight SHAP reconstruction, LIME local surrogate fidelity and SHAP-LIME agreement.",
    "actionability_design": "Equal-weight availability, numeric proximity and sparsity.",
    "standardisation_note": "This EES input preparation notebook uses the same configurable pipeline and output structure across German Credit, Pima Diabetes and COMPAS."
}

with open(LOGS_PATH / "ees_prepare_inputs_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=4)

with open(TABLES_PATH / "ees_prepare_inputs_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=4)

print("Workbook and manifest saved.")

Saved: /content/drive/MyDrive/Master_Research/07_Results/german_credit/08_01_ees_framework_prepare_inputs_german_credit/tables/ees_framework_prepare_inputs_german_credit_outputs.xlsx
Workbook and manifest saved.


## 11. Interpretation Note

This notebook creates the standardised integration layer between the explanation notebooks and the EES scoring notebooks.

It ensures that Fidelity, Actionability and later AHP scoring consume the same clean, aligned input structure across all three datasets.

In [12]:
# =========================================================
# 11. Final Completion Message
# =========================================================

print("=" * 100)
print(f"08_01 EES Prepare Inputs completed successfully for: {CONFIG['dataset_label']}")
print("=" * 100)

print(f"Tables saved to:   {TABLES_PATH}")
print(f"Datasets saved to: {DATASETS_PATH}")
print(f"Logs saved to:     {LOGS_PATH}")

print("\nKey outputs:")
print("- master_ees_input.csv")
print("- ees_model_performance_input.csv")
print("- ees_shap_reconstruction_input.csv")
print("- ees_lime_fidelity_input.csv")
print("- ees_shap_lime_agreement_input.csv")
print("- ees_actionability_input.csv")
print(f"- ees_framework_prepare_inputs_{DATASET_NAME}_outputs.xlsx")

08_01 EES Prepare Inputs completed successfully for: German Credit
Tables saved to:   /content/drive/MyDrive/Master_Research/07_Results/german_credit/08_01_ees_framework_prepare_inputs_german_credit/tables
Datasets saved to: /content/drive/MyDrive/Master_Research/07_Results/german_credit/08_01_ees_framework_prepare_inputs_german_credit/datasets
Logs saved to:     /content/drive/MyDrive/Master_Research/07_Results/german_credit/08_01_ees_framework_prepare_inputs_german_credit/logs

Key outputs:
- master_ees_input.csv
- ees_model_performance_input.csv
- ees_shap_reconstruction_input.csv
- ees_lime_fidelity_input.csv
- ees_shap_lime_agreement_input.csv
- ees_actionability_input.csv
- ees_framework_prepare_inputs_german_credit_outputs.xlsx
